# 02b · Protocol B - 50 training rounds + **50 recovery rounds**
Produces **Table III** (rounds needed to reach 95 % / 100 % of the pre-unlearning accuracy). The paper reports values up to 50
(Retrain, NoT) which cannot come from the 5-round protocol, hence this separate run. Seed 0 only ("fixed random seed" in the paper).

Resumable: finished (dataset, method, seed) results in `results/raw/` are skipped, so you can stop and re-run.
The 49 shared training rounds are trained **once per (dataset, seed)** and cached in `artifacts/checkpoints/`;
every method then starts from the same pre-unlearning model (identical to what the repo's scripts recompute each time).

Entries marked `*` in the table mean "not reached within 50 rounds" (value shown = cap).

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        break
print("project root:", ROOT)

In [ ]:
from fuguard_repro.config import IMAGE_DATASETS, METHODS, PROTOCOLS
from fuguard_repro.runner import run_grid, standard_jobs

DATASETS = IMAGE_DATASETS
SEEDS = PROTOCOLS["table3"]["seeds"]                      # (0,)
RECOVERY_ROUNDS = PROTOCOLS["table3"]["recovery_rounds"]  # 50

In [ ]:
run_grid("table3", DATASETS, standard_jobs(METHODS), SEEDS, RECOVERY_ROUNDS)

In [ ]:
from fuguard_repro.aggregate import table3_numeric, table3_df
display(table3_df(table3_numeric()))